## Setup

### General Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Dataset

Sensitive attributes include:
- sex
- race

Target variable:
- two_year_recid

In [ ]:
scores_two_years = pd.read_csv("data/compas-scores-two-years.csv")

columns_to_include = [
    "sex",
    "age",
    "age_cat",
    "race",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_charge_degree",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
    "two_year_recid",
]

dataset = scores_two_years[columns_to_include]

n = dataset.shape[0]

main_features = ["sex", "race", "two_year_recid"]

In [ ]:
categorical_columns = [
    "sex",
    "c_charge_degree",
    "two_year_recid",
    "age_cat",
    "race",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
]

In [ ]:
def display_main_features_info(df: pd.DataFrame):
    about_main_feats = {}

    for column in main_features:
        temp = pd.DataFrame()

        temp["value"] = df[column].unique().tolist()
        temp["count"] = df[column].value_counts().tolist()
        temp["percentage"] = round(
            df[column].value_counts(normalize=True) * 100, 1
        ).tolist()
        about_main_feats[column] = temp
        print(f"\n\nfeature: {column}")
        print(about_main_feats[column])

## Data Preprocessing

In [4]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
# df.info()

In [ ]:
# TODO: Handle missing values in "r_charge_degree" and "r_charge_desc" (<50% missing)
# TODO: Handle missing values in "c_charge_desc" (>1% missing)

# Encode categorical variables
df_encoded = dataset.copy()

for col in df_encoded.columns:
    if df_encoded[col].dtype == "object":
        df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

# Models

### Settings

In [ ]:
synthetic_data_count = 5000
num_epochs = 200

### Model Imports

In [ ]:
from ctgan import CTGAN
from ctgan import TVAE
from tabfairgan import TFG
from fairgan import FairGAN

## CTGAN
The CTGAN model learns the joint distribution of the dataset.

### Model Training

In [ ]:
import os

os.environ["LOKY_MAX_CPU_COUNT"] = "1"

In [ ]:
ctgan_df = dataset.copy()

ctgan_model = CTGAN(epochs=num_epochs, verbose=True)
ctgan_model.fit(ctgan_df, discrete_columns=categorical_columns)

### Synthetic Data Generation

In [ ]:
ctgan_synthetic_data = ctgan_model.sample(synthetic_data_count)

ctgan_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
ctgan_synthetic_data.to_csv("ctgan_synthetic.csv", index=False)

## TVAE

TVAE learns a latent representation of the data and reconstructs it through a decoder network.

Unlike CTGAN, TVAE optimizes a probabilistic objective rather than using adversarial training.

### Model Training

In [ ]:
tvae_df = dataset.copy()

tvae_model = TVAE(epochs=num_epochs, verbose=True)
tvae_model.fit(tvae_df, discrete_columns=categorical_columns)

### Synthetic Data Generation

In [ ]:
tvae_synthetic_data = tvae_model.sample(synthetic_data_count)

tvae_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
tvae_synthetic_data.to_csv("tvae_synthetic.csv", index=False)

## FairGAN

Our implementation of the FairGAN model can be found [here](fairgan.py)

### Model Training

In [ ]:
fairgan_df = dataset.copy()


# TODO: Make sure race can be handled
fairgan_model = FairGAN(
    fairgan_df, "sex", "two_year_recid", epochs=num_epochs, verbose=True
)
fairgan_model.train()

In [ ]:
tvae_df = dataset.copy()

tvae_model = TVAE(epochs=num_epochs, verbose=True)
tvae_model.fit(tvae_df, discrete_columns=categorical_columns)

### Synthetic Data Generation

In [ ]:
fairgan_synthetic_data = fairgan_model.generate(synthetic_data_count)

fairgan_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
fairgan_synthetic_data.to_csv("fairgan_synthetic.csv", index=False)

## TabFairGAN

TabFairGAN differs from standard generative models such as CTGAN in that fairness constraints are incorporated directly into the training process.

Instead of using a separate fit function, the model is trained using a custom training loop.

### Model Training

**Fairness Constraints**

To generate fair synthetic data, you need to pass a dictionary containing the following parameters:

- `fair_epochs`: Number of fair training epochs (integer).
- `lamda`: Lambda parameter controlling the trade-off between fairness and accuracy (float).
- `S`: Protected attribute (string, e.g., "sex").
- `Y`: Decision label (string, e.g., "income").
- `S_under`: Value representing the underprivileged group for the protected attribute (string, e.g., " Female").
- `Y_desire`: Desired value for the label (string, e.g., " >50K").

In [ ]:
tfg_df = dataset.copy()

# Define fairness configuration
fairness_config = {
    "fair_epochs": 50,
    "lamda": 0.5,
    "S": "sex",
    "Y": "income",
    "S_under": "Female",
    "Y_desire": ">50K",
}

# Initialize TabFairGAN with fairness constraints
tfg_model = TFG(
    tfg_df, epochs=200, batch_size=256, fairness_config=fairness_config
)

# Train the model
tfg_model.train()

### Synthetic Data Generation

In [ ]:
tfg_synthetic_data = tfg_model.generate_fake_df(num_rows=synthetic_data_count)

tfg_synthetic_data.head()

Export the synthetic data for analysis

In [ ]:
tfg_synthetic_data.to_csv("tabfairgan_synthetic.csv", index=False)

# Statistical Evaluation

In [ ]:
def plot_dist_comparison(df: pd.DataFrame, synth_data: pd.DataFrame):
    numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns

    for col in numerical_cols:
        plt.figure()
        sns.kdeplot(df[col], label="Real", fill=True)
        sns.kdeplot(synth_data[col], label="Synthetic", fill=True)
        plt.title(col)
        plt.legend()
        plt.show()


plot_dist_comparison(dataset, ctgan_synthetic_data)

# Fairness Analysis

We compare income outcomes across sensitive groups (e.g., sex).

## CTGAN

In [ ]:
# Convert income to binary
dataset["income_binary"] = dataset["income"].apply(
    lambda x: 1 if ">50K" in x else 0
)
ctgan_synthetic_data["income_binary"] = ctgan_synthetic_data["income"].apply(
    lambda x: 1 if ">50K" in x else 0
)

In [ ]:
# Statistical Parity
def statistical_parity(df, sensitive_col, outcome_col):
    group_rates = df.groupby(sensitive_col)[outcome_col].mean()
    return group_rates.max() - group_rates.min(), group_rates


real_spd, real_rates = statistical_parity(
    dataset, "sex_binary", "income_binary"
)
fake_spd, fake_rates = statistical_parity(
    fairgan_synthetic_data, "sex_binary", "income_binary"
)

print("REAL SPD:", real_spd)
print(real_rates)

print("\nFAKE SPD (FairGAN):", fake_spd)
print(fake_rates)

In [ ]:
# Disparity Gap
def disparate_impact(df, sensitive_col, outcome_col):
    group_rates = df.groupby(sensitive_col)[outcome_col].mean()
    return group_rates.min() / group_rates.max()


real_di = disparate_impact(dataset, "sex_binary", "income_binary")
fake_di = disparate_impact(
    fairgan_synthetic_data, "sex_binary", "income_binary"
)

print("REAL DI:", real_di)
print("FAKE DI (FairGAN):", fake_di)

## Interpretation

If the disparity in synthetic data closely matches the real data, bias is preserved.

If the disparity increases, bias is amplified. If it decreases, bias is mitigated.

These findings highlight that synthetic data may replicate societal biases present in the original dataset.

# Downstream Classifier
To evaluate the utility of the generated data, standard machine learning classifiers will be trained using the synthetic datasets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.impute import SimpleImputer

### Logistic Regression


#### Data Preprocessing

In [ ]:
# TODO: Generalise this for all models and datasets, not just TabFairGAN


# CLEAN DATA
def clean_dataframe(df):
    df = df.copy()

    # Remove leading/trailing spaces in strings
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()

    return df


# Clean datasets
real_train_df = clean_dataframe(dataset)
synthetic_df = clean_dataframe(tfg_synthetic_data)


real_train_df, real_test_df = train_test_split(
    real_train_df, test_size=0.2, random_state=42
)


def encode_target(df, target_col):
    df = df.copy()
    df[target_col] = df[target_col].astype(str)
    df[target_col] = df[target_col].str.strip()
    df[target_col] = df[target_col].str.replace(".", "", regex=False)
    df[target_col] = df[target_col].map({"<=50K": 0, ">50K": 1})
    return df


real_train_df = encode_target(real_train_df, "income")
real_test_df = encode_target(real_test_df, "income")
synthetic_df = encode_target(synthetic_df, "income")


#  BUILD PIPELINE
def build_pipeline(df, target_col):
    X = df.drop(columns=[target_col])

    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
            ("num", "passthrough", numerical_cols),
        ]
    )

    pipeline = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(max_iter=1000)),
        ]
    )

    return pipeline


# FAIRNESS METRICS
def demographic_parity(y_pred, sensitive):
    groups = sensitive.unique()
    rates = [y_pred[sensitive == g].mean() for g in groups]
    return abs(rates[0] - rates[1])


def disparate_impact(y_pred, sensitive):
    groups = sensitive.unique()
    rate_0 = y_pred[sensitive == groups[0]].mean()
    rate_1 = y_pred[sensitive == groups[1]].mean()
    return min(rate_0 / (rate_1 + 1e-6), rate_1 / (rate_0 + 1e-6))


def equal_opportunity(y_true, y_pred, sensitive):
    groups = sensitive.unique()
    tprs = []

    for g in groups:
        mask = sensitive == g
        tp = ((y_pred == 1) & (y_true == 1) & mask).sum()
        fn = ((y_pred == 0) & (y_true == 1) & mask).sum()
        tprs.append(tp / (tp + fn + 1e-6))

    return abs(tprs[0] - tprs[1])


def equalized_odds(y_true, y_pred, sensitive):
    groups = sensitive.unique()
    tpr_list, fpr_list = [], []

    for g in groups:
        mask = sensitive == g

        tp = ((y_pred == 1) & (y_true == 1) & mask).sum()
        fn = ((y_pred == 0) & (y_true == 1) & mask).sum()
        fp = ((y_pred == 1) & (y_true == 0) & mask).sum()
        tn = ((y_pred == 0) & (y_true == 0) & mask).sum()

        tpr = tp / (tp + fn + 1e-6)
        fpr = fp / (fp + tn + 1e-6)

        tpr_list.append(tpr)
        fpr_list.append(fpr)

    return abs(tpr_list[0] - tpr_list[1]) + abs(fpr_list[0] - fpr_list[1])


# TRAIN + EVALUATE
def run_tabfairgan_evaluation(train_df, test_df, target_col, sensitive_col):

    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]

    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col]
    sensitive = test_df[sensitive_col]

    model = build_pipeline(train_df, target_col)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "demographic_parity": demographic_parity(y_pred, sensitive),
        "disparate_impact": disparate_impact(y_pred, sensitive),
        "equal_opportunity": equal_opportunity(y_test, y_pred, sensitive),
        "equalized_odds": equalized_odds(y_test, y_pred, sensitive),
    }

    return results


# RUN IT
results = run_tabfairgan_evaluation(
    train_df=synthetic_df,  # train on synthetic
    test_df=real_test_df,  # test on real
    target_col="income",
    sensitive_col="sex",
)

print("TabFairGAN Results:")
print(results)